# Reconnaissance d'Émotions Faciales en Temps Réel

Ce notebook entraîne un réseau de neurones convolutif (CNN) sur le dataset **FER-2013** pour reconnaître 7 émotions faciales :
, , , , , , .

**Pipeline :**
1. Chargement et prétraitement des données
2. Architecture CNN (Conv2D + BatchNorm + Dropout)
3. Entraînement avec callbacks (EarlyStopping, ModelCheckpoint)
4. Visualisation des courbes d'apprentissage
5. Évaluation et matrice de confusion


## 1. Imports

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU disponible : {len(tf.config.list_physical_devices(chr(71)+chr(80)+chr(85))) > 0}")

## 2. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Modifiez uniquement ces variables
# ============================================================

# Chemin vers le dataset FER-2013 (relatif au dossier du notebook)
# Structure attendue :
#   fer2013/
#     train/  (Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral)
#     test/   (Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral)
BASE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
TRAIN_DIR = os.path.join(BASE_DIR, "fer2013", "train")
TEST_DIR  = os.path.join(BASE_DIR, "fer2013", "test")

# Hyperparamètres
IMAGE_SIZE  = (48, 48)
BATCH_SIZE  = 64
EPOCHS      = 30          # EarlyStopping s'en chargera si nécessaire
MODEL_PATH  = "mon_modele.keras"

# Labels FER-2013
EMOTIONS = ["Colere", "Degout", "Peur", "Joie", "Triste", "Surprise", "Neutre"]

print(f"Train dir : {TRAIN_DIR}")
print(f"Test dir  : {TEST_DIR}")
print(f"Dossier train existe : {os.path.exists(TRAIN_DIR)}")
print(f"Dossier test existe  : {os.path.exists(TEST_DIR)}")

## 3. Chargement et prétraitement des données

In [ ]:
# Data augmentation sur le train pour réduire l'overfitting
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Validation : uniquement le rescale (pas d'augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print(f"
Classes détectées : {train_generator.class_indices}")

## 4. Architecture du modèle CNN

In [ ]:
model = Sequential([
    Input(shape=(48, 48, 1)),

    # Bloc 1
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    # Bloc 2
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    # Bloc 3
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    # Classificateur
    Flatten(),
    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.5),
    Dense(7, activation="softmax")  # 7 émotions FER-2013
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Entraînement avec callbacks

In [ ]:
callbacks = [
    # Arrêt anticipé si la val_loss ne s'améliore plus pendant 5 epochs
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Sauvegarde automatique du meilleur modèle
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    # Réduction du learning rate si plateau
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks
)

## 6. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy ---
axes[0].plot(history.history["accuracy"], label="Train", linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation", linewidth=2)
axes[0].set_title("Précision par epoch", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Loss ---
axes[1].plot(history.history["loss"], label="Train", linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation", linewidth=2)
axes[1].set_title("Perte par epoch", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Courbes d'apprentissage — CNN FER-2013", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("courbes_apprentissage.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graphique sauvegardé : courbes_apprentissage.png")

## 7. Évaluation sur le jeu de test

In [ ]:
loss, accuracy = model.evaluate(test_generator, verbose=1)
print(f"
=== Résultats ===")
print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {accuracy * 100:.2f}%")

## 8. Matrice de confusion

In [ ]:
# Prédictions
Y_pred = model.predict(test_generator, verbose=1)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=EMOTIONS,
    yticklabels=EMOTIONS,
    linewidths=0.5
)
plt.title("Matrice de confusion — FER-2013", fontsize=14, fontweight="bold")
plt.ylabel("Réalité", fontsize=12)
plt.xlabel("Prédiction IA", fontsize=12)
plt.tight_layout()
plt.savefig("matrice_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

# Rapport détaillé par classe
print("
Rapport de classification :")
print(classification_report(y_true, y_pred, target_names=EMOTIONS))

---
## 9. (Avancé) Transfer Learning avec MobileNetV2

Pour aller plus loin et améliorer la précision (objectif ~65-70%),
on peut utiliser MobileNetV2 pré-entraîné sur ImageNet.
FER-2013 est en niveaux de gris (1 canal), on empile donc 3 fois le canal gris
pour obtenir une image RGB compatible avec MobileNetV2.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, Lambda, GlobalAveragePooling2D,
    Dense, Dropout, BatchNormalization
)

# --- Générateurs avec images RGB (3 canaux) pour MobileNetV2 ---
train_datagen_rgb = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
test_datagen_rgb = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_gen_rgb = train_datagen_rgb.flow_from_directory(
    TRAIN_DIR, target_size=(48, 48), color_mode="rgb",
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=True
)
test_gen_rgb = test_datagen_rgb.flow_from_directory(
    TEST_DIR, target_size=(48, 48), color_mode="rgb",
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)

print(f"Classes : {train_gen_rgb.class_indices}")

In [ ]:
# --- Architecture Transfer Learning ---
# Base MobileNetV2 pré-entraîné (sans les couches de classification)
base_model = MobileNetV2(
    input_shape=(48, 48, 3),
    include_top=False,
    weights="imagenet"
)

# Phase 1 : on gèle les poids de la base (feature extraction)
base_model.trainable = False

# Tête de classification
inputs  = Input(shape=(48, 48, 3))
x       = base_model(inputs, training=False)
x       = GlobalAveragePooling2D()(x)
x       = Dense(256, activation="relu")(x)
x       = BatchNormalization()(x)
x       = Dropout(0.5)(x)
outputs = Dense(7, activation="softmax")(x)

model_tl = Model(inputs, outputs)

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_tl.summary()

In [ ]:
# --- Phase 1 : Feature extraction (10 epochs) ---
callbacks_tl = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint("mon_modele_tl.keras",
                                        monitor="val_accuracy",
                                        save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=2, min_lr=1e-7, verbose=1)
]

print("=== Phase 1 : feature extraction (base gelée) ===")
history_tl = model_tl.fit(
    train_gen_rgb, epochs=10,
    validation_data=test_gen_rgb,
    callbacks=callbacks_tl
)

In [ ]:
# --- Phase 2 : Fine-tuning (on dégèle les dernières couches) ---
base_model.trainable = True

# On ne dégèle que les 30 dernières couches
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),   # LR plus faible pour le fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("=== Phase 2 : fine-tuning (30 dernières couches dégelées) ===")
history_ft = model_tl.fit(
    train_gen_rgb, epochs=20,
    validation_data=test_gen_rgb,
    callbacks=callbacks_tl
)

# Évaluation finale
loss_tl, acc_tl = model_tl.evaluate(test_gen_rgb, verbose=1)
print(f"\nTransfer Learning — Accuracy : {acc_tl * 100:.2f}%")